In [1]:
import pandas as pd

# 1. Load the uploaded file
# We use sep='\t' because this specific file uses "Tabs" instead of commas
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])

# 2. Check the data
print("Data successfully loaded!")
print(df.head()) # Shows the first 5 rows

# 3. Basic statistics
print("\n--- Basic Info ---")
print(df.info())

Data successfully loaded!
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

--- Basic Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [2]:
# Convert 'ham' to 0 and 'spam' to 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Check if it worked
print(df[['label', 'label_num']].head())

  label  label_num
0   ham          0
1   ham          0
2  spam          1
3   ham          0
4   ham          0


In [3]:
import re

def clean_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove special characters and numbers (keeping only letters)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

# Apply the cleaning function to all messages
df['clean_message'] = df['message'].apply(clean_text)

print("Original Message:", df['message'][2])
print("Cleaned Message: ", df['clean_message'][2])

Original Message: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Cleaned Message:  free entry in  a wkly comp to win fa cup final tkts st may  text fa to  to receive entry questionstd txt ratetcs apply overs


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Initialize the TF-IDF Vectorizer (The "Translator")
# stop_words='english' tells it to ignore common words like 'is', 'the', 'a'
tfidf = TfidfVectorizer(stop_words='english')

# 2. Transform our cleaned messages into numbers (X)
# And our labels into targets (y)
X = tfidf.fit_transform(df['clean_message'])
y = df['label_num']

# 3. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize the Brain (The Model)
model = LogisticRegression()

# 5. Training Phase: The AI studies the training data
model.fit(X_train, y_train)

# 6. Testing Phase: Let's see how it did on the "Final Exam"
y_pred = model.predict(X_test)

# 7. Print Results
print("--- Model Training Complete! ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nDetailed Report:\n", classification_report(y_test, y_pred))

--- Model Training Complete! ---
Accuracy Score: 95.96%

Detailed Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.70      0.82       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115



In [5]:
def predict_message(user_input):
    # 1. Clean the input text
    cleaned_input = clean_text(user_input)
    # 2. Convert to numbers using the SAME tfidf we used for training
    input_vector = tfidf.transform([cleaned_input])
    # 3. Predict!
    prediction = model.predict(input_vector)[0]

    if prediction == 1:
        return "🚨 WARNING: This message looks like a FRAUD!"
    else:
        return "✅ Safe: This message seems fine."

# Try it out!
test_msg = "Congratulations! You won a $1000 Walmart gift card. Click here to claim: http://bit.ly/scam"
print(f"Message: {test_msg}")
print(f"Result: {predict_message(test_msg)}")

test_msg_2 = "Hey mom, are we still meeting for dinner tonight?"
print(f"\nMessage: {test_msg_2}")
print(f"Result: {predict_message(test_msg_2)}")

Message: Congratulations! You won a $1000 Walmart gift card. Click here to claim: http://bit.ly/scam
Result: 🚨 WARNING: This message looks like a FRAUD!

Message: Hey mom, are we still meeting for dinner tonight?
Result: ✅ Safe: This message seems fine.


In [6]:
import numpy as np

def explain_prediction(user_input):
    cleaned_input = clean_text(user_input)
    input_vector = tfidf.transform([cleaned_input])
    prediction = model.predict(input_vector)[0]

    # Get the words in the message
    words = cleaned_input.split()

    # Find the "Fraud Score" for each word from our model
    feature_names = tfidf.get_feature_names_out()
    word_weights = model.coef_[0]

    # Create a dictionary of word: weight
    weights_dict = dict(zip(feature_names, word_weights))

    triggered_words = []
    for word in words:
        if word in weights_dict and weights_dict[word] > 0.5: # 0.5 is our threshold for "scammy"
            triggered_words.append(word)

    if prediction == 1:
        explanation = f"🚨 FRAUD DETECTED!\nReason: This message uses high-risk words: {', '.join(set(triggered_words))}."
        explanation += "\nAdvice: Do not click any links or share your bank OTP."
    else:
        explanation = "✅ SAFE MESSAGE.\nReason: No suspicious patterns found."

    return explanation

# Test the explanation
print(explain_prediction("WINNER! You won a prize. Click the link for your free gift!"))

🚨 FRAUD DETECTED!
Reason: This message uses high-risk words: prize, free, gift, link, winner, won.
Advice: Do not click any links or share your bank OTP.


In [7]:
!pip install deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.2 MB/s eta 0:00:00


In [8]:
from deep_translator import GoogleTranslator

def multilingual_alert(text, target_lang='hi'): # 'hi' is for Hindi, 'es' for Spanish, etc.
    translated = GoogleTranslator(source='auto', target=target_lang).translate(text)
    return translated

# Example: Get the explanation in Hindi
explanation_en = explain_prediction("URGENT: Your account is blocked. Call now.")
print("English:", explanation_en)
print("Hindi:", multilingual_alert(explanation_en, target_lang='hi'))

English: ✅ SAFE MESSAGE.
Reason: No suspicious patterns found.
Hindi: ✅ सुरक्षित संदेश।
कारण: कोई संदिग्ध पैटर्न नहीं मिला.


In [9]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [10]:
from gtts import gTTS
import IPython.display as ipd

def play_voice_alert(text, lang='en'):
    tts = gTTS(text=text, lang=lang)
    tts.save("alert.mp3")
    return ipd.Audio("alert.mp3", autoplay=True)

# Try it!
msg_to_speak = "Warning! This message is a scam. Please do not share your password."
play_voice_alert(msg_to_speak)

In [11]:
# Create a list of Indian language codes
indian_langs = ['hi', 'te', 'ta', 'mr', 'bn', 'gu']

def generate_multilingual_data(english_text):
    translations = {}
    for lang in indian_langs:
        # Translate the fraud pattern to Indian languages
        translations[lang] = GoogleTranslator(source='auto', target=lang).translate(english_text)
    return translations

# Example: Translating a common Indian bank scam
print(generate_multilingual_data("Your SBI account KYC has expired. Click here to update."))

{'hi': 'आपके SBI खाते की KYC समाप्त हो गई है. अपडेट करने के लिए यहां क्लिक करें.', 'te': 'మీ SBI ఖాతా KYC గడువు ముగిసింది. అప్\u200cడేట్ చేయడానికి ఇక్కడ క్లిక్ చేయండి.', 'ta': 'உங்கள் SBI கணக்கு KYC காலாவதியாகிவிட்டது. புதுப்பிக்க இங்கே கிளிக் செய்யவும்.', 'mr': 'तुमचे SBI खाते KYC कालबाह्य झाले आहे. अपडेट करण्यासाठी येथे क्लिक करा.', 'bn': 'আপনার এসবিআই অ্যাকাউন্টের KYC মেয়াদ শেষ হয়ে গেছে। আপডেট করতে এখানে ক্লিক করুন.', 'gu': 'તમારું SBI એકાઉન્ટ KYC સમાપ્ત થઈ ગયું છે. અપડેટ કરવા માટે અહીં ક્લિક કરો.'}


In [13]:
import tensorflow as tf

# 1. Create a simple Neural Network Brain
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(len(tfidf.get_feature_names_out()),)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 2. Convert it to TFLite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 3. Save it to your computer
with open('fraud_guardian.tflite', 'wb') as f:
    f.write(tflite_model)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Saved artifact at '/tmp/tmptem0uyyc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 8336), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140630759290640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140630759291792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140630759290448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140630759286992: TensorSpec(shape=(), dtype=tf.resource, name=None)
